# Clean up your SageMaker AI environment

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

This notebook deletes all resources that the workshop notebooks created, **without asking for confirmation**. Run the cells in order, from top to bottom. The cells delete:
- the SageMaker projects of step 5 and of the optional notebooks, with all their CloudFormation stacks: the project stack, which owns the CodeCommit repository, the CodeBuild projects, the CodePipeline pipeline, and the EventBridge rules, and the staging and production endpoint stacks that the deployment pipeline created. Then the S3 bucket of each project, which the project stack keeps on purpose.
- the endpoints, endpoint configurations, models, and monitoring schedules of steps 4 and 5 and of the optional notebooks
- the monitoring pipeline of step 4, its schedule, and its SNS topic
- the ML pipelines of step 3 and the model package groups with all model versions
- the project template and seed code of step 5, the bucket CORS policy, the domain tag, and the IAM roles stack for SageMaker projects, if notebook 05 created it
- the workshop data in S3, the MLflow app with its artifacts, the container image repository in Amazon ECR, and the SageMaker Python SDK configuration of step 3

The cells find the resources by the names that the workshop uses: the prefix `from-idea-to-prod`, the project names `deploy-...` and `mlops-...` of this domain, and the names stored with `%store`. They don't touch other resources in your account.

<div class="alert alert-warning">Set <code>DRY_RUN = True</code> in the configuration cell to only list what the notebook would delete. The default is <code>False</code>: the cells delete.</div>

<div class="alert alert-info">You don't need to run this notebook if you use an AWS-provided account in an instructor-led workshop.</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

In [ ]:
%store -r

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

---

## Configuration and helpers

In [ ]:
import os
import time

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# Set to True to list the resources without deleting them
DRY_RUN = False

WORKSHOP_PREFIX = "from-idea-to-prod"
PROJECT_PREFIXES = ("deploy-", "mlops-")

sm = boto3.client("sagemaker", config=Config(retries={"max_attempts": 10, "mode": "adaptive"}))
s3 = boto3.resource("s3")
cfn = boto3.client("cloudformation")
caller_identity = boto3.client("sts").get_caller_identity()
account_id = caller_identity["Account"]
caller_arn = caller_identity["Arn"]


def delete(description, action):
    """Print the resource and delete it, unless DRY_RUN is set. Errors are printed, not raised."""
    if DRY_RUN:
        print(f"[dry run] would delete {description}")
        return
    try:
        action()
        print(f"Deleted {description}")
    except ClientError as error:
        print(f"[WARN] {description}: {error.response['Error']['Message']}")


def role_name(arn):
    """arn:aws:sts::<account>:assumed-role/<role>/<session> or arn:aws:iam::<account>:role/<role> -> <role>"""
    resource = arn.split(":", 5)[-1].split("/")
    if resource[0] == "assumed-role":
        return resource[1]
    if resource[0] == "role":
        return resource[-1]
    return arn


def is_workshop_project(project_name):
    """A project of this domain, or one created outside Studio by the role that runs this notebook (for example CI)."""
    created_by = sm.describe_project(ProjectName=project_name).get("CreatedBy", {})
    if created_by.get("DomainId"):
        return created_by["DomainId"] == domain_id
    creator = created_by.get("IamIdentity", {}).get("Arn")
    return bool(creator) and role_name(creator) == role_name(caller_arn)


def stack_exists(name):
    try:
        return cfn.describe_stacks(StackName=name)["Stacks"][0]["StackStatus"] != "DELETE_COMPLETE"
    except ClientError:
        return False


def delete_stack(name):
    """Delete a CloudFormation stack and wait until it's gone."""
    if not stack_exists(name):
        return
    if DRY_RUN:
        print(f"[dry run] would delete the CloudFormation stack {name}")
        return
    print(f"Deleting the CloudFormation stack {name} ...")
    cfn.delete_stack(StackName=name)
    try:
        cfn.get_waiter("stack_delete_complete").wait(StackName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
        print(f"Deleted the CloudFormation stack {name}")
    except Exception:
        reason = next(
            (ev.get("ResourceStatusReason") for ev in cfn.describe_stack_events(StackName=name)["StackEvents"]
             if ev["ResourceStatus"] == "DELETE_FAILED"), "unknown reason")
        print(f"[WARN] The stack {name} wasn't deleted: {reason}")


def delete_bucket(name):
    """Delete all objects and versions in a bucket, then the bucket."""
    bucket = s3.Bucket(name)
    try:
        s3.meta.client.head_bucket(Bucket=name)
    except ClientError:
        return
    def action():
        bucket.object_versions.delete()
        bucket.objects.all().delete()
        bucket.delete()
    delete(f"the S3 bucket {name}", action)


def delete_monitoring_schedules(endpoint_name):
    for schedule in sm.list_monitoring_schedules(EndpointName=endpoint_name)["MonitoringScheduleSummaries"]:
        name = schedule["MonitoringScheduleName"]
        delete(f"the monitoring schedule {name}", lambda: sm.delete_monitoring_schedule(MonitoringScheduleName=name))
    if not DRY_RUN:
        # Deleting a schedule takes a few seconds, the endpoint can't be deleted before
        while sm.list_monitoring_schedules(EndpointName=endpoint_name)["MonitoringScheduleSummaries"]:
            time.sleep(5)


def paginate(method, key, **kwargs):
    """List all items of a SageMaker list API, also when boto3 has no paginator for it."""
    if sm.can_paginate(method):
        return [item for page in sm.get_paginator(method).paginate(**kwargs) for item in page[key]]
    items, token = [], None
    while True:
        response = getattr(sm, method)(**kwargs, **({"NextToken": token} if token else {}))
        items += response[key]
        token = response.get("NextToken")
        if not token:
            return items

---

## Delete the SageMaker projects
For each workshop project of this domain, in any status, the next cell does the steps below. A project that was created outside Studio, for example by a CI job, has no domain. The cell also includes such a project when the IAM role that created it is the role that runs this notebook.
1. deletes the monitoring schedules of the project endpoints
2. deletes the staging and production endpoint stacks that the deployment pipeline created, and waits until they're gone. The stacks own the endpoints, the endpoint configurations, and the models.
3. deletes the project. SageMaker AI deletes the project stack `SP-<account>-<project ID>-...`. The cell waits until the stack is gone.
4. deletes the S3 bucket `sagemaker-project-<project ID>` with the pipeline artifacts and the captured data. The project stack keeps the bucket when it's deleted.

The cell also deletes the stacks of projects that failed to create.

In [ ]:
projects = [
    p for p in paginate("list_projects", "ProjectSummaryList")
    if p["ProjectName"].startswith(PROJECT_PREFIXES)
    and p["ProjectStatus"] not in ("DeleteInProgress", "DeleteCompleted")
    and is_workshop_project(p["ProjectName"])
]
print(f"Workshop projects of the domain {domain_id} or the role {role_name(caller_arn)}: {[p['ProjectName'] for p in projects]}")

for p in projects:
    name, project_id = p["ProjectName"], p["ProjectId"]
    print(f"\n=== Project {name} ({project_id}, {p['ProjectStatus']})")
    details = sm.describe_project(ProjectName=name)

    # 1. Monitoring schedules of the project endpoints
    for endpoint in (f"{name}-staging", f"{name}-prod"):
        delete_monitoring_schedules(endpoint)

    # 2. Endpoint stacks that the deployment pipeline created
    for stage in ("deploy-staging", "deploy-prod"):
        delete_stack(f"sagemaker-{name}-{project_id}-{stage}")

    # 3. The project and its stack
    project_stack = (
        details.get("TemplateProviderDetails", [{}])[0].get("CfnTemplateProviderDetail", {})
        .get("StackDetail", {}).get("Id")
        or details.get("ServiceCatalogProvisionedProductDetails", {}).get("ProvisionedProductId")
    )
    delete(f"the project {name}", lambda: sm.delete_project(ProjectName=name))
    if not DRY_RUN:
        while True:
            try:
                status = sm.describe_project(ProjectName=name)["ProjectStatus"]
            except ClientError:
                break
            if status == "DeleteCompleted":
                break
            if status == "DeleteFailed":
                print(f"[WARN] The project {name} failed to delete: {sm.describe_project(ProjectName=name).get('ProjectStatusMessage', '')}")
                break
            time.sleep(10)
    if project_stack and project_stack.startswith("arn:aws:cloudformation"):
        delete_stack(project_stack)  # no-op if SageMaker AI already deleted it

    # 4. The project bucket, kept by the project stack
    delete_bucket(f"sagemaker-project-{project_id}")

### Delete leftover project stacks
A project that failed to create can leave its stack in the status `ROLLBACK_FAILED` or `DELETE_FAILED`, also after the project is deleted. The next cell deletes the stacks `SP-<account>-...` of the project templates of this workshop, the endpoint stacks `sagemaker-deploy-...` and `sagemaker-mlops-...`, and the project buckets, if their project doesn't exist anymore.

In [ ]:
import re

existing_project_ids = {p["ProjectId"] for p in paginate("list_projects", "ProjectSummaryList")}
workshop_templates = ("DeploymentOnlyProject", "MLOpsTemplate")

for page in cfn.get_paginator("list_stacks").paginate():
    for stack in page["StackSummaries"]:
        name, status = stack["StackName"], stack["StackStatus"]
        match = re.search(r"(p-[a-z0-9]{12})", name)
        if status == "DELETE_COMPLETE" or not match or match.group(1) in existing_project_ids:
            continue
        is_project_stack = name.startswith(f"SP-{account_id}-") and name.endswith(workshop_templates)
        is_endpoint_stack = name.startswith(("sagemaker-deploy-", "sagemaker-mlops-")) and name.endswith(("-deploy-staging", "-deploy-prod"))
        if is_project_stack or is_endpoint_stack:
            delete_stack(name)
            delete_bucket(f"sagemaker-project-{match.group(1)}")

---

## Delete the endpoints, endpoint configurations, and models
The next cell deletes the endpoints with the prefix `from-idea-to-prod` and their monitoring schedules, for example the endpoint of step 4. Then it deletes the endpoint configurations and models with the same prefix. The endpoints of the projects were deleted with their stacks.

In [ ]:
endpoints = [ep["EndpointName"] for ep in paginate("list_endpoints", "Endpoints", NameContains=WORKSHOP_PREFIX)
             if ep["EndpointName"].startswith(WORKSHOP_PREFIX)]
for endpoint in endpoints:
    delete_monitoring_schedules(endpoint)
    delete(f"the endpoint {endpoint}", lambda ep=endpoint: sm.delete_endpoint(EndpointName=ep))

if not DRY_RUN:
    for endpoint in endpoints:
        try:
            sm.get_waiter("endpoint_deleted").wait(EndpointName=endpoint)
        except Exception as error:
            print(f"[WARN] {endpoint}: {error}")

for config in paginate("list_endpoint_configs", "EndpointConfigs", NameContains=WORKSHOP_PREFIX):
    if config["EndpointConfigName"].startswith(WORKSHOP_PREFIX):
        delete(f"the endpoint configuration {config['EndpointConfigName']}",
               lambda c=config["EndpointConfigName"]: sm.delete_endpoint_config(EndpointConfigName=c))

for model in paginate("list_models", "Models", NameContains=WORKSHOP_PREFIX):
    if model["ModelName"].startswith(WORKSHOP_PREFIX):
        delete(f"the model {model['ModelName']}", lambda m=model["ModelName"]: sm.delete_model(ModelName=m))

# Monitoring schedules with the workshop prefix that belong to endpoints that don't exist anymore
for schedule in paginate("list_monitoring_schedules", "MonitoringScheduleSummaries", NameContains=WORKSHOP_PREFIX):
    delete(f"the monitoring schedule {schedule['MonitoringScheduleName']}",
           lambda s=schedule["MonitoringScheduleName"]: sm.delete_monitoring_schedule(MonitoringScheduleName=s))

---

## Delete the monitoring pipeline of step 4
The EventBridge schedule, the SageMaker pipeline, and the SNS topic that notebook 04 created. If you ran the clean-up section of notebook 04, they're already deleted.

In [ ]:
monitoring_pipeline_name = "evidently-monitoring-pipeline"
scheduler = boto3.client("scheduler")
sns = boto3.client("sns")

# Notebook 04 creates the schedule with this name. Delete it by name, because the role may not list schedules.
schedule_name = f"{monitoring_pipeline_name}-daily"
try:
    scheduler.get_schedule(Name=schedule_name)
    delete(f"the schedule {schedule_name}", lambda: scheduler.delete_schedule(Name=schedule_name))
except ClientError as error:
    if error.response["Error"]["Code"] != "ResourceNotFoundException":
        print(f"[WARN] Can't check the schedule {schedule_name}: {error.response['Error']['Message']}")

if paginate("list_pipelines", "PipelineSummaries", PipelineNamePrefix=monitoring_pipeline_name):
    delete(f"the pipeline {monitoring_pipeline_name}", lambda: sm.delete_pipeline(PipelineName=monitoring_pipeline_name))

topic_arn = f"arn:aws:sns:{region}:{account_id}:sagemaker-evidently-drift-alerts"
try:
    sns.get_topic_attributes(TopicArn=topic_arn)
    delete(f"the SNS topic {topic_arn}", lambda: sns.delete_topic(TopicArn=topic_arn))
except ClientError:
    pass

---

## Delete the ML pipelines and the model package groups
The pipelines and the model package groups with the prefix `from-idea-to-prod`, from step 3 and the optional notebooks. Each model package group loses all its model versions first. The deletion of a model version is asynchronous, so the cell waits until the group is empty before it deletes the group.

In [ ]:
for pipeline in paginate("list_pipelines", "PipelineSummaries", PipelineNamePrefix=WORKSHOP_PREFIX):
    delete(f"the pipeline {pipeline['PipelineName']}", lambda p=pipeline["PipelineName"]: sm.delete_pipeline(PipelineName=p))

for group in paginate("list_model_package_groups", "ModelPackageGroupSummaryList", NameContains=WORKSHOP_PREFIX):
    group_name = group["ModelPackageGroupName"]
    if not group_name.startswith(WORKSHOP_PREFIX):
        continue
    for package in paginate("list_model_packages", "ModelPackageSummaryList", ModelPackageGroupName=group_name):
        delete(f"the model version {package['ModelPackageArn']}",
               lambda arn=package["ModelPackageArn"]: sm.delete_model_package(ModelPackageName=arn))
    if not DRY_RUN:
        # Deleting a model version is asynchronous, the group can't be deleted while it still lists versions
        for _ in range(24):
            if not paginate("list_model_packages", "ModelPackageSummaryList", ModelPackageGroupName=group_name):
                break
            time.sleep(5)
    delete(f"the model package group {group_name}", lambda g=group_name: sm.delete_model_package_group(ModelPackageGroupName=g))

---

## Delete the project template and seed code of step 5
Notebook 05 uploaded the project template and the seed code to the default bucket, added a CORS policy to the bucket, and tagged the domain with the template location. If notebook 05 created the stack `sagemaker-projects-roles-policies` with the IAM roles for SageMaker projects, the cell deletes it too. In an AWS-provided account the roles belong to another stack, which the cell doesn't touch.

In [ ]:
for key in ["templates/deployment-only-project-template.yaml", "project-seed-code/deployment-seed-code.zip"]:
    delete(f"s3://{bucket_name}/{key}", lambda k=key: s3.Object(bucket_name, k).delete())

delete(f"the CORS policy of {bucket_name}", lambda: s3.meta.client.delete_bucket_cors(Bucket=bucket_name))

domain_arn = sm.describe_domain(DomainId=domain_id)["DomainArn"]
delete(f"the tag sagemaker:projectS3TemplatesLocation of {domain_id}",
       lambda: sm.delete_tags(ResourceArn=domain_arn, TagKeys=["sagemaker:projectS3TemplatesLocation"]))

# Delete the roles only after all projects are gone, the project stacks use them for their deletion
delete_stack("sagemaker-projects-roles-policies")

---

## Delete the workshop data in S3
All objects under the prefix `from-idea-to-prod/` of the default bucket: the datasets, the model artifacts, the captured data, the reports, and the job code that `@remote` and `@step` uploaded.

In [ ]:
for prefix in [f"{WORKSHOP_PREFIX}/", f"{WORKSHOP_PREFIX}-"]:
    objects = list(s3.Bucket(bucket_name).objects.filter(Prefix=prefix).limit(1))
    if objects:
        delete(f"all objects under s3://{bucket_name}/{prefix}",
               lambda p=prefix: s3.Bucket(bucket_name).objects.filter(Prefix=p).delete())

---

## Delete the MLflow app
Notebook [`00-start-here`](00-start-here.ipynb) created the MLflow app and stored its name in `mlflow_name`. The cell deletes the app and, if the app stores its artifacts under `s3://<default bucket>/mlflow`, as when notebook 00 created it, the artifacts too. An existing MLflow app with another artifact store keeps its artifacts.

In [ ]:
try:
    app = sm.describe_mlflow_app(Arn=mlflow_arn)
except (NameError, ClientError):
    app = None

if app and app.get("Status") not in ("Deleting", "Deleted"):
    artifact_store = app.get("ArtifactStoreUri", "")
    delete(f"the MLflow app {app.get('Name', mlflow_arn)}", lambda: sm.delete_mlflow_app(Arn=mlflow_arn))
    if artifact_store.rstrip("/") == f"s3://{bucket_name}/mlflow":
        delete(f"the MLflow artifacts in {artifact_store}",
               lambda: s3.Bucket(bucket_name).objects.filter(Prefix="mlflow/").delete())
else:
    print("No MLflow app to delete")

---

## Delete the container image in Amazon ECR
The repository `sagemaker-xgboost-byoc` with all its images, which notebook 00 built. The endpoints and jobs that use the image are deleted already.

In [ ]:
ecr = boto3.client("ecr")
try:
    ecr_repo_name = xgboost_image.split("/")[-1].split(":")[0]
except NameError:
    ecr_repo_name = "sagemaker-xgboost-byoc"

try:
    ecr.describe_repositories(repositoryNames=[ecr_repo_name])
    delete(f"the ECR repository {ecr_repo_name}", lambda: ecr.delete_repository(repositoryName=ecr_repo_name, force=True))
except ecr.exceptions.RepositoryNotFoundException:
    print(f"The ECR repository {ecr_repo_name} doesn't exist")

---

## Delete the SageMaker Python SDK configuration of step 3
Notebook 03 copied `config.yaml` with the defaults of the `@step` decorator to the user-level configuration folder of the SageMaker Python SDK.

In [ ]:
from platformdirs import user_config_dir

sdk_config = os.path.join(user_config_dir("sagemaker"), "config.yaml")
if os.path.exists(sdk_config):
    delete(sdk_config, lambda: os.remove(sdk_config))

---

## What the notebook doesn't delete
- the CloudWatch log groups `/aws/codebuild/sagemaker-<project name>-...` of the project CodeBuild projects. Delete them in the CloudWatch console if you want.
- the CloudWatch logs of the training and processing jobs, in the shared log groups `/aws/sagemaker/TrainingJobs` and `/aws/sagemaker/ProcessingJobs`. They don't cost anything until you store a lot of data. Delete the log streams with the prefix `from-idea-to-prod` in the CloudWatch console if you want.
- the SageMaker AI domain, the user profile, and the JupyterLab space. If you created them for this workshop, delete them as described in the [README](README.md#clean-up).

## Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>